# Big Data Analysis of Google Play Store Applications / Group 22
###### Team Members: Lotta Kauppinen, Jenna Kiviaho, Marjaana Koski, Jani Laakso, Aleksi Savukoski

#### Dataset from Kaggle: Google Play Store Apps, created by Gautham Prakash
https://www.kaggle.com/datasets/gauthamp10/google-playstore-apps

#### This notebook explains how the data was cleaned and preprocessed for the analysis.


Google Play Store Applications dataset contains over 2.3 million rows. The dataset includes multiple variables to describe applications' popularity, metrics, monetization and features. The CSV file is processed with pyspark and stored into MongoDB, and studied for big data analysis with Spark SQL and DataFrame API. Finally, insights were visualized using Python libraries and compared with MongoDB aggregation results to demonstrate big‑data processing workflows across multiple technologies.

In [24]:
#  Spark Session:

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PlaystoreAnalysis") \
    .config("spark.jars.packages", 
            "org.mongodb.spark:mongo-spark-connector_2.12:10.3.0") \
    .config("spark.mongodb.read.connection.uri", 
            "mongodb://127.0.0.1:27017") \
    .config("spark.mongodb.write.connection.uri", 
            "mongodb://127.0.0.1:27017") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()


In [25]:
# Due to e.g. commas, escape quotes and other special characters, careful CSV parsing is required
df = (spark.read.option("header", "true").option("inferSchema", "true").option("quote", "\"").option("escape", "\"").option("multiLine", "true").csv("data/Google-Playstore.csv"))
df.printSchema()

root
 |-- App Name: string (nullable = true)
 |-- App Id: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Rating: double (nullable = true)
 |-- Rating Count: integer (nullable = true)
 |-- Installs: string (nullable = true)
 |-- Minimum Installs: long (nullable = true)
 |-- Maximum Installs: long (nullable = true)
 |-- Free: boolean (nullable = true)
 |-- Price: double (nullable = true)
 |-- Currency: string (nullable = true)
 |-- Size: string (nullable = true)
 |-- Minimum Android: string (nullable = true)
 |-- Developer Id: string (nullable = true)
 |-- Developer Website: string (nullable = true)
 |-- Developer Email: string (nullable = true)
 |-- Released: string (nullable = true)
 |-- Last Updated: string (nullable = true)
 |-- Content Rating: string (nullable = true)
 |-- Privacy Policy: string (nullable = true)
 |-- Ad Supported: boolean (nullable = true)
 |-- In App Purchases: boolean (nullable = true)
 |-- Editors Choice: boolean (nullable = true)
 |-- Scr

In [26]:
# The original amount of rows and columns:
print("Rows:", df.count())
print("Columns:", len(df.columns))

Rows: 2312944
Columns: 24


## Cleaning the data

The unprocessed dataset contains null values, possible duplicates, wrong field formants, and unnecessary columns for data analysis.

Irrelevant columns, which cannot be utilized in data analysis, containing e.g. personnel information were removed. Fields like Released, Min and Max Installs, and Size were fixed to match the correct data format. Complete rows containing null values in relevant fields were dropped out. As App Id is the unique column key it was validated, that this column does not have duplicate ids. As popularity is one of the most imporant features in this study, all columns with value 0 Rating or Rating count were converted to null values.

In [ ]:
# Checking the amount of null values by column:
from pyspark.sql.functions import col, when, count

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+--------+------+--------+------+------------+--------+----------------+----------------+----+-----+--------+----+---------------+------------+-----------------+---------------+--------+------------+--------------+--------------+------------+----------------+--------------+------------+
|App Name|App Id|Category|Rating|Rating Count|Installs|Minimum Installs|Maximum Installs|Free|Price|Currency|Size|Minimum Android|Developer Id|Developer Website|Developer Email|Released|Last Updated|Content Rating|Privacy Policy|Ad Supported|In App Purchases|Editors Choice|Scraped Time|
+--------+------+--------+------+------------+--------+----------------+----------------+----+-----+--------+----+---------------+------------+-----------------+---------------+--------+------------+--------------+--------------+------------+----------------+--------------+------------+
|       0|     0|       0| 22883|       22883|     107|             107|               0|   0|    0|     135| 196|           6530|      

### Dropping columns that are not needed

In [28]:
# Due to the purpose of this analysis, information of these columns is not needed: 
# Developer Website, Developer Email, Privacy Policy, Scraped Time, so let's drop them.
# Currency has multiple values, so that should remain as it affects the Price field.
df = df.drop("Developer Website","Developer Email","Privacy Policy","Scraped Time")

### Changing the data types

In [29]:
# Checking the data types
df.printSchema()

root
 |-- App Name: string (nullable = true)
 |-- App Id: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Rating: double (nullable = true)
 |-- Rating Count: integer (nullable = true)
 |-- Installs: string (nullable = true)
 |-- Minimum Installs: long (nullable = true)
 |-- Maximum Installs: long (nullable = true)
 |-- Free: boolean (nullable = true)
 |-- Price: double (nullable = true)
 |-- Currency: string (nullable = true)
 |-- Size: string (nullable = true)
 |-- Minimum Android: string (nullable = true)
 |-- Developer Id: string (nullable = true)
 |-- Released: string (nullable = true)
 |-- Last Updated: string (nullable = true)
 |-- Content Rating: string (nullable = true)
 |-- Ad Supported: boolean (nullable = true)
 |-- In App Purchases: boolean (nullable = true)
 |-- Editors Choice: boolean (nullable = true)



In [30]:
from pyspark.sql.functions import translate, when, round, expr, lower

# Changing data types
df = df.withColumn("Installs", expr("try_cast(translate(Installs, '+,', '') as long)")) # Removing characters '+' and ','
df = df.withColumn("Price", expr("try_cast(Price as float)"))

# Handling column "Size"
df = df.withColumn("Size", 
    when(col("Size").contains("k"),
         expr("try_cast(translate(Size, 'k,', ' .') as float)") / 1024) # Removing 'k' and ',', then divided by 1024 so every size is in MB
    .when(col("Size").contains("M"), 
         expr("try_cast(translate(Size, 'M,', ' .') as float)")) # Removing characters 'M' and ','
    .otherwise(expr("try_cast(Size as float)")) # If it's only a number and no characters
)

# Rounding column "Size"
df = df.withColumn("Size", round(col("Size"), 2))

# Checking data types again to make sure they are correct
print(df.schema["Installs"].dataType)
print(df.schema["Price"].dataType)
print(df.schema["Size"].dataType)


LongType()
FloatType()
DoubleType()


In [31]:
# Minimum Installs and Maximum Installs to long
df = df.withColumn("Minimum Installs", expr("try_cast(`Minimum Installs` as long)"))
df = df.withColumn("Maximum Installs", expr("try_cast(`Maximum Installs` as long)"))

# Confirming changes
print(df.schema["Minimum Installs"].dataType)
print(df.schema["Maximum Installs"].dataType)

LongType()
LongType()


In [32]:
from pyspark.sql.functions import to_date

# Changing dates into yyyy, MM, DD format
df = df.withColumn("Released", to_date(col("Released"), "MMM d, yyyy")) \
       .withColumn("Last Updated", to_date(col("Last Updated"), "MMM d, yyyy"))

# Checking the outcome
df.select("Released", "Last Updated").show(20)
print(df.schema["Released"].dataType)
print(df.schema["Last Updated"].dataType)


+----------+------------+
|  Released|Last Updated|
+----------+------------+
|2020-02-26|  2020-02-26|
|2020-05-21|  2021-05-06|
|2019-08-09|  2019-08-19|
|2018-09-10|  2018-10-13|
|2020-02-21|  2018-11-12|
|2018-12-24|  2019-12-20|
|2019-09-23|  2019-09-27|
|2019-06-21|  2019-06-21|
|      NULL|  2018-12-07|
|2019-09-22|  2020-10-07|
|2020-07-30|  2020-07-30|
|2018-01-10|  2018-06-27|
|2018-04-03|  2021-06-11|
|2020-02-09|  2021-05-14|
|2018-09-05|  2020-05-30|
|2020-04-05|  2021-03-23|
|2016-11-28|  2019-10-30|
|2019-04-24|  2019-05-05|
|2020-07-01|  2021-05-26|
|2020-12-26|  2021-03-23|
+----------+------------+
only showing top 20 rows

DateType()
DateType()


In [33]:
# Confirm the data types after fixes
# Finalized formats
df.printSchema()

root
 |-- App Name: string (nullable = true)
 |-- App Id: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Rating: double (nullable = true)
 |-- Rating Count: integer (nullable = true)
 |-- Installs: long (nullable = true)
 |-- Minimum Installs: long (nullable = true)
 |-- Maximum Installs: long (nullable = true)
 |-- Free: boolean (nullable = true)
 |-- Price: float (nullable = true)
 |-- Currency: string (nullable = true)
 |-- Size: double (nullable = true)
 |-- Minimum Android: string (nullable = true)
 |-- Developer Id: string (nullable = true)
 |-- Released: date (nullable = true)
 |-- Last Updated: date (nullable = true)
 |-- Content Rating: string (nullable = true)
 |-- Ad Supported: boolean (nullable = true)
 |-- In App Purchases: boolean (nullable = true)
 |-- Editors Choice: boolean (nullable = true)



### Dropping columns with null values

There are more than 1 million rows with rating value as 0. Then again, these rows do not have rating count above 0. Earlier it was noted, that there are approximately 22 883 rows where both rating and rating count are null.

To convert the rows with zero ratings and null ratings to a single representation, the 0 ratings and counts are converted to nulls. This way the non rated apps remain as null values, as 0 is not a rating that can be given for an app.

In [34]:
# Confirm, that if the rating is 0, also the rating count is 0, and vice versa
print(f'Rows with null ratings and rating count: {df.filter((col("Rating").isNull()) & (col("Rating Count").isNull())).count()}')
print(f'Rows with rating value as 0: {df.filter(col("Rating") == 0).count()}')
print(f'Rows with rating value as 0 and rating count > 0 : {df.filter((col("Rating") == 0) & (col("Rating Count") > 0)).count()}')
print(f'Rows with rating value > 0 and rating count as 0 : {df.filter((col("Rating") > 0) & (col("Rating Count") == 0)).count()}')
print(f'Rows with rating value as not null and rating count as null: {df.filter((col("Rating").isNotNull()) & (col("Rating Count").isNull())).count()}')
print(f'Rows with rating value as null and rating count as not null: {df.filter((col("Rating").isNull()) & (col("Rating Count").isNotNull())).count()}')

Rows with null ratings and rating count: 22883
Rows with rating value as 0: 1059762
Rows with rating value as 0 and rating count > 0 : 0
Rows with rating value > 0 and rating count as 0 : 0
Rows with rating value as not null and rating count as null: 0
Rows with rating value as null and rating count as not null: 0


In [35]:
#Change the 0 rating rows to null ratings
df = df.withColumn("Rating",when((col("Rating") == 0) & (col("Rating Count") == 0), None).otherwise(col("Rating")))

df = df.withColumn("Rating Count",when((col("Rating Count") == 0) & (col("Rating").isNull()), None).otherwise(col("Rating Count")))

In [36]:
# Now all 0 ratings and rating counts are consistently null
df.filter((col("Rating").isNull()) & (col("Rating Count").isNull())).count()

1082645

We retained Size and Released columns despite their ~6% null rate. Removing them would narrow the data significantly and could have potentially biased the analysis of other attributes like popularity and categories.

As we want to study the popularitity against multiple variables, rows below containing null values are dropped out. Out of these columns, the column installs is the most important one, as application without installations cannot have reviews or meaningful data. In addition to column installations, remaining columns contain information where null values are anomalities and therefore these rows are dropped. This diminishes the dataset only by ~0.29 % while making the data cleaner.

In [37]:
df_null= df.na.drop(subset=['Installs'])
df_no_null = df_null.na.drop(subset=['Installs', 'Minimum Installs', 'Ad Supported', 'Developer Id', 'In App Purchases', 'Free', 'Minimum Android', 'Currency'])

print(f'Original row quantity: {df.count()}')
print(f'Row quantity after removing null installs: {df_null.count()}')
print(f'Row quantity after removing other null rows: {df_no_null.count()}')
print(f'Change-% in row quantity after removing null rows: {((df.count()-df_no_null.count())/df.count()*100)}')

Original row quantity: 2312944
Row quantity after removing null installs: 2312837
Row quantity after removing other null rows: 2306246
Change-% in row quantity after removing null rows: 0.2895876424158994


In [38]:
# Drop the columns above
df = df.na.drop(subset=['Installs', 'Minimum Installs', 'Ad Supported', 'Developer Id', 'In App Purchases', 'Free', 'Minimum Android', 'Currency'])

df.count()

2306246

#### Handling Duplicates

Due to the nature of this data, we need to check the rows that have the same App ID. Because there are no such rows, nothing needs to be deleted.

In [39]:
from pyspark.sql.functions import count
df.groupBy("App Id").count().filter("count > 1").show()

+------+-----+
|App Id|count|
+------+-----+
+------+-----+



## Data storage, indexing, and schema in MongoDB

### Data schema corrections, structure handling and indexing before Mongo DB upload


Before uploading the cleaned dataframe to MongoDB, field names are changed to snake_case.

The schema design part consists of categorizing relevant features into same group instead of uploading the file as such as a flat format. Additionally, indexing is performed for the most queried fields.

Finally, the dataset is uploaded to MongoDB.

#### Define schema

In [40]:
# Use snake_case for data storage

df = df.select(
    col("App Name").alias("app_name"),
    col("App Id").alias("app_id"),
    col("Category").alias("category"),
    col("Rating").alias("rating"),
    col("Rating Count").alias("rating_count"),
    col("Installs").alias("installs"),
    col("Minimum Installs").alias("min_installs"),
    col("Maximum Installs").alias("max_installs"),
    col("Free").alias("free"),
    col("Price").alias("price"),
    col("Currency").alias("currency"),
    col("Size").alias("size_mb"),
    col("Minimum Android").alias("min_android"),
    col("Developer Id").alias("developer_id"),
    col("Released").alias("released"),
    col("Last Updated").alias("last_updated"),
    col("Content Rating").alias("content_rating"),
    col("Ad Supported").alias("ads_supported"),
    col("In App Purchases").alias("in_app_purchases"),
    col("Editors Choice").alias("editors_choice")
)

In [41]:
# Define schema from flat to structured groups

from pyspark.sql.functions import struct, col

df_schema = df.select(
    "app_id",
    "app_name",
    "category",
    "developer_id",

    struct(
        "rating",
        "rating_count",
        "installs",
        "min_installs",
        "max_installs"
    ).alias("popularity_metrics"),

    struct(
        col("free").alias("is_free"),
        col("price"),
        col("currency")
    ).alias("pricing"),

    struct(
        col("size_mb"),
        col("min_android"),
        col("content_rating")
    ).alias("details"),

    struct(
        col("ads_supported"),
        col("in_app_purchases"),
        col("editors_choice")
    ).alias("features"),

    struct(
        col("released"),
        col("last_updated")
    ).alias("app_development")
)

### Upload schema to MongoDB

In [42]:
# Upload cleaned, grouped dataframe to MongoDB
df_schema.write \
    .format("mongodb") \
    .mode("overwrite") \
    .option("database", "playstore") \
    .option("collection", "apps") \
    .save()

#### Create and confirm indexes

In [43]:
from pymongo import MongoClient
client = MongoClient("mongodb://localhost:27017/")
db = client.playstore

In [44]:
# With indexing, the frequently queried columns can be processed more efficiently.
# The indexes are added to following columns

db.apps.create_index([("app_id", 1)])
db.apps.create_index([("app_name", 1)])
db.apps.create_index([("category", 1)])
db.apps.create_index([("pricing.is_free", 1)])

# Popularity metrics in ascending order
db.apps.create_index([("popularity_metrics.rating", -1)])


'popularity_metrics.rating_-1'

In [45]:
# Run the list of existing indexes
list(db.apps.list_indexes())

[SON([('v', 2), ('key', SON([('_id', 1)])), ('name', '_id_')]),
 SON([('v', 2), ('key', SON([('app_id', 1)])), ('name', 'app_id_1')]),
 SON([('v', 2), ('key', SON([('app_name', 1)])), ('name', 'app_name_1')]),
 SON([('v', 2), ('key', SON([('category', 1)])), ('name', 'category_1')]),
 SON([('v', 2), ('key', SON([('pricing.is_free', 1)])), ('name', 'pricing.is_free_1')]),
 SON([('v', 2), ('key', SON([('popularity_metrics.rating', -1)])), ('name', 'popularity_metrics.rating_-1')])]